# Lezione — SOLUZIONI: EDA e programmazione di base per scomposizione

**Idea guida.** Prima di scrivere codice, si scompone il problema in sotto-problemi sempre più piccoli,
finché ogni foglia dell'albero è abbastanza semplice da diventare **una funzione** (o poche righe).
In project management si chiama *Work Breakdown Structure*; in informatica si parla di
**scomposizione funzionale**, *top-down design* o *stepwise refinement* (Wirth, 1971).

Per ogni esercizio:
1. leggi la domanda;
2. completa/discuti l'**albero di scomposizione** (in markdown, prima del codice);
3. scrivi **una funzione per ogni foglia**, poi una funzione "radice" che le chiama in ordine.

**Dataset** (nella stessa cartella del notebook):
- `penguins.csv` — Palmer Penguins: 344 pinguini, 3 specie, misure di becco, pinna e massa (Antartide, 2007–2009).
- `titanic.csv` — 891 passeggeri del Titanic con classe, sesso, età, tariffa e sopravvivenza.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

penguins = pd.read_csv("penguins.csv")
titanic = pd.read_csv("titanic.csv")

## Esercizio 1 — Il dataset è pronto per l'analisi? (penguins)

**Domanda:** posso fidarmi di `penguins.csv` così com'è?

```
Il dataset è pronto?
├── Quanto è grande?            → numero di righe e colonne
├── Che tipo ha ogni colonna?   → numeriche vs categoriche
├── Ci sono valori mancanti?    → conteggio e % per colonna
└── Ci sono duplicati?          → numero di righe duplicate
```

**Da fare:** scrivi `report_qualita(df)` che restituisce un DataFrame con, per ogni colonna:
`tipo`, `n_mancanti`, `perc_mancanti`, `n_valori_unici`; e stampa forma e numero di duplicati.

*Domanda di discussione:* le righe con `sex` mancante vanno buttate? E quelle con tutte le misure mancanti?

In [ ]:
def report_qualita(df):
    print(f"Forma: {df.shape[0]} righe x {df.shape[1]} colonne")
    print(f"Righe duplicate: {df.duplicated().sum()}")
    report = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_mancanti": df.isna().sum(),
        "perc_mancanti": (df.isna().mean() * 100).round(1),
        "n_valori_unici": df.nunique(),
    })
    return report

report_qualita(penguins)

## Esercizio 2 — Statistiche descrittive per gruppo (penguins)

**Domanda:** quanto pesano i pinguini delle diverse specie, e c'è differenza tra maschi e femmine?

```
Confronto della massa corporea
├── Pulizia     → togli le righe senza massa o senza sesso
├── Raggruppa   → per specie (e poi per specie + sesso)
├── Riassumi    → n, media, mediana, deviazione standard
└── Presenta    → tabella ordinata e arrotondata
```

**Da fare:** scrivi `riassunto_per_gruppo(df, gruppi, colonna)` riutilizzabile con qualsiasi colonna numerica
(prova con `body_mass_g` e poi con `flipper_length_mm`).

*Domanda di discussione:* media e mediana sono simili? Cosa ti dice questo sulla forma della distribuzione?

In [ ]:
def pulisci(df, colonne):
    return df.dropna(subset=colonne)

def riassunto_per_gruppo(df, gruppi, colonna):
    dati = pulisci(df, gruppi + [colonna])
    tab = (dati.groupby(gruppi)[colonna]
                .agg(["count", "mean", "median", "std"])
                .round(1)
                .sort_values("mean", ascending=False))
    return tab

display(riassunto_per_gruppo(penguins, ["species"], "body_mass_g"))
display(riassunto_per_gruppo(penguins, ["species", "sex"], "body_mass_g"))

## Esercizio 3 — Guardare prima di concludere (penguins)

**Domanda:** la lunghezza del becco e la profondità del becco sono correlate? In che verso?

```
Relazione tra due misure
├── Correlazione globale        → un numero su tutto il dataset
├── Correlazione per specie     → un numero per ogni specie
└── Grafico                     → scatter plot colorato per specie
```

**Da fare:** scrivi `correlazione(df, x, y, gruppo=None)` e `scatter_per_gruppo(df, x, y, gruppo)`,
poi confronta i risultati con `x="bill_length_mm"`, `y="bill_depth_mm"`, `gruppo="species"`.

*Domanda di discussione:* il segno della correlazione globale è uguale a quello dentro ogni specie?
Come si chiama questo fenomeno? (suggerimento: inizia con "S")

In [ ]:
def correlazione(df, x, y, gruppo=None):
    dati = df.dropna(subset=[x, y])
    if gruppo is None:
        return round(dati[x].corr(dati[y]), 2)
    return dati.groupby(gruppo).apply(lambda g: round(g[x].corr(g[y]), 2))

def scatter_per_gruppo(df, x, y, gruppo):
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for nome, g in df.groupby(gruppo):
        ax.scatter(g[x], g[y], label=nome, alpha=0.7, s=18)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.legend(title=gruppo)
    plt.show()

print("Globale:", correlazione(penguins, "bill_length_mm", "bill_depth_mm"))
print(correlazione(penguins, "bill_length_mm", "bill_depth_mm", "species"))
scatter_per_gruppo(penguins, "bill_length_mm", "bill_depth_mm", "species")
# Globale ≈ -0.24, dentro ogni specie ≈ +0.4/+0.65: paradosso di Simpson.

## Esercizio 4 — Chi è sopravvissuto? (titanic)

**Domanda:** la probabilità di sopravvivere dipendeva da sesso e classe?

```
Tasso di sopravvivenza
├── Per sesso                  → % sopravvissuti
├── Per classe                 → % sopravvissuti
├── Per sesso × classe         → tabella a doppia entrata
└── Grafico                    → barre raggruppate
```

**Da fare:** scrivi `tasso_sopravvivenza(df, gruppi)` che restituisce la percentuale di sopravvissuti
(la colonna `survived` vale 0/1: qual è il trucco per avere la percentuale?) e una funzione che disegna
il grafico a barre della tabella sesso × classe.

*Domanda di discussione:* il dataset ha colonne ridondanti (`survived`/`alive`, `pclass`/`class`,
`embarked`/`embark_town`). Perché è un problema, e quale tieni?

In [ ]:
def tasso_sopravvivenza(df, gruppi):
    # la media di una colonna 0/1 è la proporzione di 1
    return (df.groupby(gruppi)["survived"].mean() * 100).round(1)

def barre_sopravvivenza(df, righe, colonne):
    tab = tasso_sopravvivenza(df, [righe, colonne]).unstack()
    ax = tab.plot(kind="bar", figsize=(6, 4), rot=0)
    ax.set_ylabel("% sopravvissuti"); ax.set_ylim(0, 100)
    plt.show()
    return tab

display(tasso_sopravvivenza(titanic, ["sex"]))
display(tasso_sopravvivenza(titanic, ["class"]))
barre_sopravvivenza(titanic, "class", "sex")

## Esercizio 5 — Una piccola pipeline completa (titanic)

**Domanda:** i bambini si sono salvati più degli adulti? E viaggiare da soli aiutava?

Questa volta **l'albero lo disegni tu** (in una cella markdown) prima di scrivere codice. Vincoli:
- l'età manca per circa il 20% dei passeggeri: decidi cosa fare e scrivilo nell'albero;
- crea le colonne `fascia_eta` (es. 0–12, 13–17, 18–39, 40–59, 60+) e `dimensione_famiglia` (`sibsp + parch + 1`);
- la funzione radice `analisi(df)` deve chiamare in ordine: `pulisci` → `aggiungi_variabili` → `riassumi`
  e non contenere altra logica.

*Domanda di discussione:* se riempi le età mancanti con la mediana, cosa succede alla fascia
in cui cade la mediana? Cambia la conclusione?

In [ ]:
# Albero:
# analisi
# ├── pulisci            → tieni solo colonne utili, togli righe senza età (scelta dichiarata)
# ├── aggiungi_variabili → fascia_eta (pd.cut), dimensione_famiglia, solo
# └── riassumi           → tasso di sopravvivenza per fascia_eta e per dimensione_famiglia

def pulisci(df):
    colonne = ["survived", "sex", "pclass", "age", "sibsp", "parch", "fare"]
    return df[colonne].dropna(subset=["age"]).copy()

def aggiungi_variabili(df):
    df = df.copy()
    df["fascia_eta"] = pd.cut(df["age"], bins=[0, 12, 17, 39, 59, 120],
                              labels=["0-12", "13-17", "18-39", "40-59", "60+"])
    df["dimensione_famiglia"] = df["sibsp"] + df["parch"] + 1
    df["solo"] = df["dimensione_famiglia"] == 1
    return df

def riassumi(df):
    per_eta = df.groupby("fascia_eta", observed=True)["survived"].agg(["count", "mean"])
    per_famiglia = df.groupby("dimensione_famiglia")["survived"].agg(["count", "mean"])
    return {"per_eta": per_eta.round(2), "per_famiglia": per_famiglia.round(2)}

def analisi(df):
    return riassumi(aggiungi_variabili(pulisci(df)))

risultati = analisi(titanic)
display(risultati["per_eta"])
display(risultati["per_famiglia"])